# Ham or Spam?

🎯 The goal of this challenge is to classify emails as spams (1) or normal emails (0)

🧹 First, you will apply cleaning techniques to these textual data

👩🏻‍🔬 Then, you will convert the cleaned texts into a numerical representation

✉️ Eventually, you will apply the ***Multinomial Naive Bayes*** model to classify each email as either a spam or a regular email.

## (0) The NTLK library (Natural Language Toolkit)

In [1]:
# !pip install nltk

In [2]:
# When importing nltk for the first time, we need to also download a few built-in libraries

import nltk

nltk.download('stopwords')
nltk.download('punkt')      # For nltk<3.9.0
nltk.download('punkt_tab')  # For nltk>=3.9.0
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/Adrien/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/Adrien/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/Adrien/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/Adrien/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/Adrien/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
import pandas as pd

df = pd.read_csv("https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/10-Natural-Language-Processing/ham_spam_emails.csv")
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


## (1) Cleaning the (text) dataset

The dataset is made up of emails that are classified as ham [0] or spam[1]. You need to clean the dataset before training a prediction model.

### (1.1) Remove Punctuation

❓ Create a function to remove the punctuation. Apply it to the `text` column and add the output to a new column in the dataframe called `clean_text` ❓

In [4]:
df.rename(columns={'text': 'clean_text'}, inplace=True)

In [5]:
import string

def remove_punctuation(text_string):
    if isinstance(text_string, str):
        return ''.join(char for char in text_string if char not in string.punctuation)
    return text_string

df['clean_text'] = df['clean_text'].apply(remove_punctuation)
df.head()

,clean_text,spam
0,Subject naturally irresistible your corporate ...,1
1,Subject the stock trading gunslinger fanny is...,1
2,Subject unbelievable new homes made easy im w...,1
3,Subject 4 color printing special request addi...,1
4,Subject do not have money get software cds fr...,1


### (1.2) Lower Case

❓ Create a function to lowercase the text. Apply it to `clean_text` ❓

In [6]:
def case_lowering(string):
    if isinstance(string, str):
        return string.lower()
    return string

df['clean_text'] = df['clean_text'].apply(case_lowering)
df.head()

,clean_text,spam
0,subject naturally irresistible your corporate ...,1
1,subject the stock trading gunslinger fanny is...,1
2,subject unbelievable new homes made easy im w...,1
3,subject 4 color printing special request addi...,1
4,subject do not have money get software cds fr...,1


### (1.3) Remove Numbers

❓ Create a function to remove numbers from the text. Apply it to `clean_text` ❓

In [7]:
def remove_digits(string):
    if isinstance(string, str):
        return ''.join(char for char in string if not char.isdigit())
    return string

df['clean_text'] = df['clean_text'].apply(remove_digits)
df.head()

,clean_text,spam
0,subject naturally irresistible your corporate ...,1
1,subject the stock trading gunslinger fanny is...,1
2,subject unbelievable new homes made easy im w...,1
3,subject color printing special request addit...,1
4,subject do not have money get software cds fr...,1


### (1.4) Remove StopWords

❓ Create a function to remove stopwords from the text. Apply it to `clean_text`. ❓

In [8]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(string):
    if isinstance(string, str):
        words = string.split()
        stopwords_removed = [w for w in words if w.lower() not in stop_words]
        return ' '.join(stopwords_removed)
    return string

df['clean_text'] = df['clean_text'].apply(remove_stopwords)
df.head()

,clean_text,spam
0,subject naturally irresistible corporate ident...,1
1,subject stock trading gunslinger fanny merrill...,1
2,subject unbelievable new homes made easy im wa...,1
3,subject color printing special request additio...,1
4,subject money get software cds software compat...,1


### (1.5) Lemmatize

❓ Create a function to lemmatize the text. Make sure the output is a single string, not a list of words. Apply it to `clean_text`. ❓

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(string):
    if isinstance(string, str):

        words = string.split()

        lemmatized_verbs = [lemmatizer.lemmatize(word, pos='v') for word in words]

        lemmatized_nouns = [lemmatizer.lemmatize(word, pos='n') for word in lemmatized_verbs]

        return ' '.join(lemmatized_nouns)

    return string

df['clean_text'] = df['clean_text'].apply(lemmatize_text)
df.head()

,clean_text,spam
0,subject naturally irresistible corporate ident...,1
1,subject stock trade gunslinger fanny merrill m...,1
2,subject unbelievable new home make easy im wan...,1
3,subject color print special request additional...,1
4,subject money get software cd software compati...,1


## (2) Bag-of-words Modelling

### (2.1) Digitizing the textual data into numbers

❓ Vectorize the `clean_text` to a Bag-of-Words representation with a default CountVectorizer. Save as `X_bow`. ❓

In [17]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer()
X_bow = count_vectorizer.fit_transform(df['clean_text'])
X_bow.toarray()
count_vectorizer.get_feature_names_out()

array(['aa', 'aaa', 'aaaenerfax', ..., 'zzn', 'zzncacst', 'zzzz'],
      dtype=object)

In [20]:
import pandas as pd

vectorized_strings = pd.DataFrame(X_bow.toarray(), columns=count_vectorizer.get_feature_names_out(),
                                  index=df['clean_text'])

vectorized_strings.head()

,aa,aaa,aaaenerfax,aadedeji,aagrawal,aal,aaldous,aaliyah,aall,aanalysis,...,zwzm,zxghlajf,zyban,zyc,zygoma,zymg,zzmacmac,zzn,zzncacst,zzzz
clean_text,,,,,,,,,,,,,,,,,,,,,
subject naturally irresistible corporate identity lt really hard recollect company market full suqgestions information isoverwhelminq good catchy logo stylish statlonery outstanding website make task much easier promise havinq order iogo company automaticaily become world ieader isguite ciear without good product effective business organization practicable aim hotat nowadays market promise market effort become much effective list clear benefit creativeness hand make original logo specially do reflect distinctive company image convenience logo stationery provide format easy use content management system letsyou change website content even structure promptness see logo draft within three business day affordability market break make gap budget satisfaction guarantee provide unlimited amount change extra fee surethat love result collaboration look portfolio interest,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
subject stock trade gunslinger fanny merrill muzo colza attainder penultimate like esmark perspicuous ramble segovia group try sling kansa tanzania yes chameleon continuant clothesman libretto chesapeake tight waterway herald hawthorn like chisel morristown superior deoxyribonucleic clockwork try hall incredible mcdougall yes hepburn einsteinian earmark sapling boar duane plain palfrey inflexible like huzzah pepperoni bedtime nameable attire try edt chronography optimum yes pirogue diffusion albeit,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
subject unbelievable new home make easy im want show homeowner pre approve home loan fix rate offer extend unconditionally credit way factor take advantage limit time opportunity ask visit website complete minute post approval form look foward hear dorcas pittman,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
subject color print special request additional information click click printable version order form pdf format phone fax e mail ramsey goldengraphix com request additional information click click printable version order form pdf format golden graphix print azusa canyon rd irwindale ca e mail message advertisement solicitation,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
subject money get software cd software compatibility great grow old along best yet tradgedies finish death comedy end marriage,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### (2.2) Multinomial Naive Bayes Modelling

❓ Cross-validate a MultinomialNB model with the bag-of-words data. Score the model's accuracy. ❓

In [22]:
import numpy as np

from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import recall_score

# Feature/Target

X = df['clean_text'] # On ne met pas X_bow puisque X_bow a été obtenu en passant déjà une première fois df['clean_text'] par le CountVectorizer
y = df['spam']

# Pipeline vectorizer + Naive Bayes

pipeline_naive_bayes = make_pipeline(CountVectorizer(), MultinomialNB())

# Cross-validation

cv_results = cross_validate(pipeline_naive_bayes, X, y, cv=5, scoring=['recall'])
average_recall = cv_results['test_recall'].mean()
np.round(average_recall,2)

0.98

🏁 Congratulations !

💾 Don't forget to git add/commit/push your notebook...

🚀 ... and move on to the next challenge !